# Tomahawk 16 T pressure-coupled ballooning test

Attach **Tomahawk_16T_Ballooning_Test_Input.zip** as one Kaggle dataset and enable Internet for the pinned DESC install. Run all cells on a CPU notebook. The notebook locates the ZIP whether Kaggle mounts it as an archive or extracts it.

This replaces the old hand-set `alpha/alpha_crit < 1` proxy with an ideal-ballooning eigenvalue and a Newcomb metric calculated from a pressure-coupled equilibrium. It checks the 6.1 GWth/core, 1.273 MPa input profile on the 16-period CAD Fourier boundary. The 6.127 kPa CSV edge offset is removed for the equilibrium so no unmodeled edge sheet current is implied; the pressure gradient is unchanged. The q profile is even about the magnetic axis: `q(rho)=1.2+(q96-1.2)*(rho**2/0.96**2)**power`.

**Scope:** The coil MGRID, free boundary, engineering current limit, low-mode MHD, and plant efficiency are separate checks. A fixed-boundary eigenvalue screen cannot certify the reactor or net electric output. Results are zipped after each case so a time limit leaves a usable checkpoint.


In [1]:
import importlib.util, subprocess, sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError('DESC requires Python >= 3.10')
if importlib.util.find_spec('desc') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                    'desc-opt==0.17.3'], check=True)
import desc, numpy, scipy, netCDF4, jax
if desc.__version__ != '0.17.3':
    raise RuntimeError(f'Expected desc-opt 0.17.3, got {desc.__version__}')
print('Python', sys.version.split()[0], 'DESC', desc.__version__,
      'NumPy', numpy.__version__, 'SciPy', scipy.__version__, 'JAX', jax.__version__)


INFO: pip is looking at multiple versions of lineax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 167.7 kB/s eta 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flax 0.12.10 requires jax>=0.11.1, but you have jax 0.9.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Python 3.13.15 DESC 0.17.3 NumPy 2.1.3 SciPy 1.16.3 JAX 0.9.2


In [2]:
import hashlib, json, os, shutil, zipfile
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'ballooning_kaggle_smoke'
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'ballooning_input'
DATA.mkdir(exist_ok=True)
roots = [p for p in (Path('/kaggle/input'), Path.cwd()) if p.is_dir()]
archives = [p for root in roots for p in root.rglob('Tomahawk_16T_Ballooning_Test_Input.zip')]
if archives:
    with zipfile.ZipFile(sorted(set(archives))[0]) as z:
        required = {'combined_6p1_profile.csv', 'desc_pressure_ballooning_test.py', 'input_manifest.json'}
        if not required.issubset(z.namelist()):
            raise ValueError(f'Input ZIP missing {sorted(required-set(z.namelist()))}')
        for name in required:
            (DATA / name).write_bytes(z.read(name))
else:
    manifests = [p for root in roots for p in root.rglob('input_manifest.json')
                 if (p.parent / 'combined_6p1_profile.csv').exists()
                 and (p.parent / 'desc_pressure_ballooning_test.py').exists()]
    if not manifests:
        raise FileNotFoundError('Attach Tomahawk_16T_Ballooning_Test_Input.zip as a Kaggle dataset')
    for name in ('combined_6p1_profile.csv', 'desc_pressure_ballooning_test.py', 'input_manifest.json'):
        shutil.copyfile(sorted(set(manifests))[0].parent / name, DATA / name)
manifest = json.loads((DATA / 'input_manifest.json').read_text())
for name, expected in manifest['input_sha256'].items():
    actual = hashlib.sha256((DATA / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Input checksum mismatch for {name}')
print('Validated pressure input and engine:', DATA)
print('MGRID reference digest:', manifest['assembled_16T_MGRID_sha256_for_later_free_boundary_test'])


Validated pressure input and engine: /kaggle/working/ballooning_input
MGRID reference digest: afb07089205aa3d2cdfa03d53b48298ae5092f23c0e70db72938c21f59a78013


In [3]:
import csv, json, os, subprocess, sys, time, zipfile
from pathlib import Path

RESULTS = WORK / 'ballooning_fix_results'
RESULTS.mkdir(exist_ok=True)
ZIP = WORK / 'Tomahawk_16T_Ballooning_Fix_Results.zip'
CASES = [
    ('q4_even_L12', 4.0, 2.0, 12),
    ('q3p5_even_L14', 3.5, 2.0, 14),
    ('q3_even_L14', 3.0, 2.0, 14),
    ('q3_even_L16', 3.0, 2.0, 16),
    ('q3_even_L18', 3.0, 2.0, 18),
]
# For local integration checks only; Kaggle runs every case by default.
if os.environ.get('TOMAHAWK_TEST_CASES'):
    wanted = set(os.environ['TOMAHAWK_TEST_CASES'].split(','))
    CASES = [case for case in CASES if case[0] in wanted]
    if not CASES:
        raise ValueError('TOMAHAWK_TEST_CASES names no test case')
deadline = time.monotonic() + 11 * 3600
completed = []

def checkpoint():
    q3 = {r['name']: r for r in completed if r.get('ok') and r['name'].startswith('q3_even_')}
    grades = [q3.get(f'q3_even_L{n}') for n in (14, 16, 18)]
    screen = (all(grades) and all(r['ideal_ballooning_screen_pass']
              and r['max_ballooning_lambda'] <= 0 and r['min_newcomb_metric'] > 0
              and r['rms_force_over_rms_JB_plus_gradp'] < 0.001 for r in grades))
    summary = {
        'status': 'FIXED_BOUNDARY_BALLOONING_SCREEN_PASS' if screen else 'INCOMPLETE_OR_FAILED_SCREEN',
        'not_a_free_boundary_or_plant_stability_approval': True,
        'source_manifest_sha256': manifest['source_manifest_sha256'],
        'mgrid_reference_sha256_not_coupled': manifest['assembled_16T_MGRID_sha256_for_later_free_boundary_test'],
        'cases': completed,
    }
    (RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False) + '\n')
    with (RESULTS / 'summary.csv').open('w', newline='') as stream:
        columns = ['name','ok','q96','qpower','resolution','max_ballooning_lambda',
                   'min_newcomb_metric','ideal_ballooning_screen_pass',
                   'rms_force_over_rms_JB_plus_gradp','edge_current_A','error']
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows({key: row.get(key) for key in columns} for row in completed)
    tmp = ZIP.with_suffix('.tmp')
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as out:
        for path in sorted(RESULTS.glob('*')):
            if path.is_file():
                out.write(path, path.name)
        out.write(DATA / 'input_manifest.json', 'input_manifest.json')
    tmp.replace(ZIP)

checkpoint()
for name, q96, qpower, resolution in CASES:
    if deadline - time.monotonic() < 600:
        completed.append({'name': name, 'ok': False, 'error': 'Stopped before 12-hour Kaggle limit'})
        checkpoint()
        break
    dest = RESULTS / f'{name}.json'
    log = RESULTS / f'{name}.log'
    command = [sys.executable, '-u', str(DATA / 'desc_pressure_ballooning_test.py'),
               str(DATA / 'combined_6p1_profile.csv'), '--out', str(dest),
               '--resolution', str(resolution), '--maxiter', '80',
               '--q96', str(q96), '--qpower', str(qpower)]
    env = dict(os.environ)
    env.setdefault('JAX_PLATFORM_NAME', 'cpu')
    env.setdefault('MPLCONFIGDIR', str(WORK / 'mpl_cache'))
    env.setdefault('OPENBLAS_NUM_THREADS', '4')
    print('Running', name, flush=True)
    try:
        with log.open('w') as stream:
            subprocess.run(command, env=env, stdout=stream, stderr=subprocess.STDOUT,
                           timeout=max(1, int(deadline-time.monotonic()-300)), check=True)
        detail = json.loads(dest.read_text())
        completed.append({'name': name, 'ok': True, 'q96': q96, 'qpower': qpower,
                          'resolution': resolution,
                          'max_ballooning_lambda': detail['max_ballooning_lambda'],
                          'min_newcomb_metric': detail['min_newcomb_metric'],
                          'ideal_ballooning_screen_pass': detail['ideal_ballooning_screen_pass'],
                          'rms_force_over_rms_JB_plus_gradp': detail['rms_force_over_rms_JB_plus_gradp'],
                          'edge_current_A': detail['current_at_rho_0p96_and_edge_A'][-1]})
        print(name, 'lambda=', detail['max_ballooning_lambda'],
              'Newcomb=', detail['min_newcomb_metric'], flush=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError, ValueError, KeyError) as exc:
        completed.append({'name': name, 'ok': False, 'error': f'{type(exc).__name__}: {exc}'})
        print(name, 'failed; read its .log inside results ZIP', flush=True)
    finally:
        checkpoint()
print('Saved:', ZIP)
print((RESULTS / 'summary.csv').read_text())


Running q4_even_L12
q4_even_L12 lambda= 0.0002013629601225294 Newcomb= -0.09359249880175659
Running q3p5_even_L14
q3p5_even_L14 lambda= -0.00014368402643787204 Newcomb= 0.002043375245820895
Running q3_even_L14
q3_even_L14 lambda= -0.001019222627079723 Newcomb= 0.014985689715533861
Running q3_even_L16
q3_even_L16 lambda= -0.0009884456405670557 Newcomb= 0.014573608838834051
Running q3_even_L18
q3_even_L18 lambda= -0.0009856439824832974 Newcomb= 0.014455116097574322
Saved: /kaggle/working/Tomahawk_16T_Ballooning_Fix_Results.zip
name,ok,q96,qpower,resolution,max_ballooning_lambda,min_newcomb_metric,ideal_ballooning_screen_pass,rms_force_over_rms_JB_plus_gradp,edge_current_A,error
q4_even_L12,True,4.0,2.0,12,0.0002013629601225294,-0.09359249880175659,False,0.0007894766022139692,22594440.65544913,
q3p5_even_L14,True,3.5,2.0,14,-0.00014368402643787204,0.002043375245820895,True,0.0005278415043594841,25463198.229279697,
q3_even_L14,True,3.0,2.0,14,-0.001019222627079723,0.014985689715533861,True

## How to read the result

Use the CSV and logs in `Tomahawk_16T_Ballooning_Fix_Results.zip`. A negative maximum ideal-ballooning eigenvalue and positive Newcomb metric at all sampled surfaces are the local high-mode screen. The summary only reports a pass when all three q=3.0 resolution cases pass and the independent-grid force residual ratio is under 0.001. The current is the model's required plasma current, not a validated coil setting. The coil MGRID, free-boundary fit, signed current, low-mode stability and plant output remain separate.
